
## 1) Load & clean metadata


In [1]:

import re
import pandas as pd

try:
    from num2words import num2words  # used only for numbers >= 1 billion
except ImportError:
    num2words = None

BASE_URL = "https://huggingface.co/datasets/KickItLikeShika/NileTTS-dataset/resolve/main/"


def load_metadata(filename: str) -> pd.DataFrame:
    df = pd.read_csv(BASE_URL + filename, sep="|")
    df.columns = df.columns.str.strip()
    return df


def clean_whitespace(text) -> str:
    """Trim and collapse repeated whitespace. Does not touch the words."""
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()


def clean_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["text"] = df["text"].astype("string").map(clean_whitespace)
    df["audio_file"] = df["audio_file"].astype("string").str.strip()
    df["speaker_name"] = df["speaker_name"].astype("string").str.strip()
    return df


In [2]:
train_df = load_metadata("metadata_train.csv")
eval_df = load_metadata("metadata_eval.csv")

print("Training shape:", train_df.shape)
print("Evaluation shape:", eval_df.shape)
print("Columns:", train_df.columns.tolist())
print("Missing values:\n", train_df.isna().sum())
print("Duplicate texts:", train_df["text"].duplicated().sum())
print("Speakers:\n", train_df["speaker_name"].value_counts())

clean_train_df = clean_metadata(train_df)
clean_eval_df = clean_metadata(eval_df)
print("Empty texts after cleaning:", clean_train_df["text"].eq("").sum())

Training shape: (8571, 3)
Evaluation shape: (950, 3)
Columns: ['audio_file', 'text', 'speaker_name']
Missing values:
 audio_file      0
text            0
speaker_name    0
dtype: int64
Duplicate texts: 2
Speakers:
 speaker_name
SPEAKER_01    4385
SPEAKER_02    4186
Name: count, dtype: int64
Empty texts after cleaning: 0


## 2) Digit unification

In [3]:
_DIGIT_MAP = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")


def unify_digits(text: str) -> str:
    text = text.translate(_DIGIT_MAP)
    text = text.replace("٫", ".")  # Arabic decimal separator
    # Thousands separator: 1,500 -> 1500
    return re.sub(r"(?<=\d),(?=\d{3}(?!\d))", "", text)


print(unify_digits("عندي ١٥ سنة والسعر ١,٥٠٠ و٧٫٥"))


عندي 15 سنة والسعر 1500 و7.5


## 3) Numbers → Egyptian words

In [4]:

UNITS = ["صفر", "واحد", "اتنين", "تلاتة", "أربعة", "خمسة", "ستة", "سبعة", "تمانية", "تسعة"]
TEENS = {
    10: "عشرة", 11: "إحداشر", 12: "إتناشر", 13: "تلتاشر", 14: "أربعتاشر",
    15: "خمستاشر", 16: "ستاشر", 17: "سبعتاشر", 18: "تمنتاشر", 19: "تسعتاشر",
}
TENS = {
    2: "عشرين", 3: "تلاتين", 4: "أربعين", 5: "خمسين",
    6: "ستين", 7: "سبعين", 8: "تمانين", 9: "تسعين",
}
HUNDREDS = {
    1: "مية", 2: "ميتين", 3: "تلتمية", 4: "ربعمية", 5: "خمسمية",
    6: "ستمية", 7: "سبعمية", 8: "تمنمية", 9: "تسعمية",
}


def _below_100(n: int) -> str:
    if n < 10:
        return UNITS[n]
    if n < 20:
        return TEENS[n]
    tens, unit = divmod(n, 10)
    return TENS[tens] if unit == 0 else f"{UNITS[unit]} و{TENS[tens]}"


def _below_1000(n: int) -> str:
    hundreds, rest = divmod(n, 100)
    parts = []
    if hundreds:
        parts.append(HUNDREDS[hundreds])
    if rest:
        parts.append(_below_100(rest))
    return " و".join(parts)


def _scaled(n: int, singular: str, dual: str, plural: str, general: str) -> str:
    """Words for a count of thousands / millions."""
    if n == 1:
        return singular
    if n == 2:
        return dual
    if 3 <= n <= 10:
        return f"{UNITS[n]} {plural}"
    return f"{_below_1000(n)} {general}"


def number_to_egyptian_words(number: int) -> str:
    if number < 0:
        return "ناقص " + number_to_egyptian_words(-number)
    if number >= 1_000_000_000:  # rare: fall back to standard Arabic
        if num2words is None:
            raise ValueError("num2words is required for numbers >= 1 billion")
        return num2words(number, lang="ar")

    millions, rest = divmod(number, 1_000_000)
    thousands, rest = divmod(rest, 1000)

    parts = []
    if millions:
        parts.append(_scaled(millions, "مليون", "مليونين", "ملايين", "مليون"))
    if thousands:
        parts.append(_scaled(thousands, "ألف", "ألفين", "آلاف", "ألف"))
    if rest or not parts:
        parts.append(_below_1000(rest) if rest else UNITS[0])
    return " و".join(parts)


def decimal_to_egyptian_words(text: str) -> str:
    """'7.5' -> 'سبعة فاصلة خمسة'. Leading zeros / long tails are read digit by digit."""
    whole, frac = text.split(".")
    whole_words = number_to_egyptian_words(int(whole))
    if frac.startswith("0") or len(frac) > 2:
        frac_words = " ".join(UNITS[int(d)] for d in frac)
    else:
        frac_words = number_to_egyptian_words(int(frac))
    return f"{whole_words} فاصلة {frac_words}"


## 4) Time & date → Egyptian words

- **الوقت:** `7:00` سبعة، `7:15` سبعة وربع، `7:30` سبعة ونص، `7:45` تمانية إلا ربع، `7:20` سبعة وتلت، `7:40` تمانية إلا تلت. الساعة بتتحول لنظام 12.
- **التاريخ:** لازم يكون 3 أجزاء. `15/10/2026` أو `15-10-26` (يوم/شهر/سنة)، و`2026-10-22` (سنة/شهر/يوم). أي حاجة مش منطقية (شهر 13 مثلًا) بتتساب زي ما هي.

In [5]:
MONTHS = {
    1: "يناير", 2: "فبراير", 3: "مارس", 4: "أبريل", 5: "مايو", 6: "يونيو",
    7: "يوليو", 8: "أغسطس", 9: "سبتمبر", 10: "أكتوبر", 11: "نوفمبر", 12: "ديسمبر",
}


def _minutes_words(m: int) -> str:
    words = number_to_egyptian_words(m)
    if m % 5 == 0:
        return words
    if m == 1:
        return "دقيقة"
    if m == 2:
        return "دقيقتين"
    return f"{words} دقايق" if m <= 10 else f"{words} دقيقة"


def time_to_egyptian_words(hour: int, minute: int):
    """Return Egyptian words for HH:MM, or None if the time is not valid."""
    if not (0 <= hour <= 24 and 0 <= minute <= 59):
        return None
    h = number_to_egyptian_words(hour % 12 or 12)
    nxt = number_to_egyptian_words((hour + 1) % 12 or 12)

    special = {0: h, 15: f"{h} وربع", 20: f"{h} وتلت", 30: f"{h} ونص",
               40: f"{nxt} إلا تلت", 45: f"{nxt} إلا ربع"}
    if minute in special:
        return special[minute]
    if minute < 30:
        return f"{h} و{_minutes_words(minute)}"
    return f"{nxt} إلا {_minutes_words(60 - minute)}"


def date_to_egyptian_words(a: str, b: str, c: str):
    """y/m/d if the first part has 4 digits, otherwise d/m/y. None if not a valid date."""
    if len(a) == 4:
        year, month, day = int(a), int(b), int(c)
    else:
        day, month, year = int(a), int(b), int(c)
    if year < 100:
        year += 2000
    if not (1 <= month <= 12 and 1 <= day <= 31):
        return None
    return f"{number_to_egyptian_words(day)} {MONTHS[month]} {number_to_egyptian_words(year)}"

## 5) One regex for all numbers

In [6]:
_NUMBER_RE = re.compile(
    r"(?P<date>\d{1,4}[/-]\d{1,2}(?:[/-]\d{1,4})?)"
    r"|(?P<time>\d{1,2}:\d{2})"
    r"|(?P<percent>\d+(?:\.\d+)?)\s*%"
    r"|(?P<decimal>\d+\.\d+)"
    r"|(?P<integer>\d+)"
)


def _replace_number(match: re.Match) -> str:
    if match.group("date") or match.group("time"):
        return match.group(0)  # left for a dedicated date/time step
    if match.group("percent"):
        value = match.group("percent")
        words = decimal_to_egyptian_words(value) if "." in value else number_to_egyptian_words(int(value))
        return f"{words} في المية"
    if match.group("decimal"):
        return decimal_to_egyptian_words(match.group("decimal"))
    return number_to_egyptian_words(int(match.group("integer")))


def expand_numbers(text: str) -> str:
    return _NUMBER_RE.sub(_replace_number, text)


def normalize_egyptian_text(text: str) -> str:
    text = clean_whitespace(text)
    text = unify_digits(text)
    text = expand_numbers(text)
    return clean_whitespace(text)

## 6) Tests

In [7]:

# Save cleaned metadata files

clean_train_df.to_csv(
    "cleaned_metadata_train.csv",
    sep="|",
    index=False,
    encoding="utf-8-sig"
)

clean_eval_df.to_csv(
    "cleaned_metadata_eval.csv",
    sep="|",
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned metadata files saved successfully!")


Cleaned metadata files saved successfully!


In [8]:
samples = [
    "نسبة النجاح وصلت إلى 90%.",
    "المخاطرة وصلت إلى 5%.",
    "النسبة وصلت إلى 7.5%.",
    "النسبة وصلت إلى 12.5%.",
    "النسبة وصلت إلى 200%.",
    "السعر 250 جنيه.",
    "الشقة بـ 1,500,000 جنيه.",
    "أنا عندي ١٥ سنة.",
    "التاريخ 15/10/2026.",
    "هتروح الساعة 7:30؟",
    "إزيك؟  عامل إيه؟",
]

for s in samples:
    print("Original  :", s)
    print("Normalized:", normalize_egyptian_text(s))
    print("-" * 50)

Original  : نسبة النجاح وصلت إلى 90%.
Normalized: نسبة النجاح وصلت إلى تسعين في المية.
--------------------------------------------------
Original  : المخاطرة وصلت إلى 5%.
Normalized: المخاطرة وصلت إلى خمسة في المية.
--------------------------------------------------
Original  : النسبة وصلت إلى 7.5%.
Normalized: النسبة وصلت إلى سبعة فاصلة خمسة في المية.
--------------------------------------------------
Original  : النسبة وصلت إلى 12.5%.
Normalized: النسبة وصلت إلى إتناشر فاصلة خمسة في المية.
--------------------------------------------------
Original  : النسبة وصلت إلى 200%.
Normalized: النسبة وصلت إلى ميتين في المية.
--------------------------------------------------
Original  : السعر 250 جنيه.
Normalized: السعر ميتين وخمسين جنيه.
--------------------------------------------------
Original  : الشقة بـ 1,500,000 جنيه.
Normalized: الشقة بـ مليون وخمسمية ألف جنيه.
--------------------------------------------------
Original  : أنا عندي ١٥ سنة.
Normalized: أنا عندي خمستاشر سنة.
----------

## 7) Apply to the dataset and save

In [9]:
for df in (clean_train_df, clean_eval_df):
    df["text_normalized"] = df["text"].map(normalize_egyptian_text)

changed = clean_train_df[clean_train_df["text"] != clean_train_df["text_normalized"]]
print("Rows changed by normalization:", len(changed), "of", len(clean_train_df))
print(changed[["text", "text_normalized"]].head(10).to_string(index=False))

clean_train_df.to_csv("cleaned_metadata_train.csv", sep="|", index=False, encoding="utf-8-sig")
clean_eval_df.to_csv("cleaned_metadata_eval.csv", sep="|", index=False, encoding="utf-8-sig")
print("Saved cleaned metadata files.")


Rows changed by normalization: 451 of 8571
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 text                                                                                                                                                                                                                                                                                                                                                                                                                                                        

#**Speech Dataset Preparation**